# Climate Crop Yield Intelligence 🌾🌍

I'M data scientist engineer and one of my clients is a company in the agriculture industry that wants to understand how climate change is affecting crop yield.

The client is not asking only for charts. They want to know **which crops are more climate sensitive, where the risk is higher, and if management factors like irrigation or fertilizer can help reduce that risk.**

This project is a full rebuild of an older university idea using updated public data and stronger methodology.

## Problem Statement

The main objective is to analyze how temperature, precipitation, irrigation and fertilizer use relate to crop yield across countries and crop types.

I also want to find hidden patterns that can help the client answer practical questions:

- Which crops look more sensitive to warmer years?
- Is there one "best temperature", or does the response change by crop?
- Does more rainfall always mean better yield?
- Do irrigation and fertilizer appear to reduce climate exposure?
- Can a model predict future-period yield better than a simple crop baseline?

**IMPORTANT NOTE:** Correlation is not causation. I will use correlation as a signal, not as proof.

## 1. Importing Libraries

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# make the project package available when running from /notebooks
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from climate_crop_yield.data import build_analysis_panel
from climate_crop_yield.features import (
    add_features,
    add_temperature_bins,
    crop_temperature_sensitivity,
    validate_panel,
)
from climate_crop_yield.model import fit_time_split
from climate_crop_yield.plots import PASTEL_10, label_bars, set_project_style

set_project_style()
pd.set_option("display.max_columns", 30)

## 2. Dataset Overview

For V1 I use public country-year data from established sources:

- **Crop yield:** FAO crop production statistics, distributed through Our World in Data.
- **Temperature:** ERA5 / Copernicus annual surface temperature.
- **Precipitation:** ERA5 / Copernicus annual precipitation.
- **Fertilizer:** FAO via World Bank / Our World in Data.
- **Irrigation:** FAO via World Bank / Our World in Data.

The analysis focuses on **1990–2023** because this gives a useful overlap between climate, management and crop data.

### Crops in V1
Wheat, Maize, Rice, Potatoes, Soybeans and Barley.

In [ ]:
# this cell downloads the latest source data
df = build_analysis_panel(start_year=1990, end_year=2023)
validate_panel(df)

print("Rows:", f"{len(df):,}")
print("Countries:", df["Code"].nunique())
print("Crops:", df["crop"].nunique())
print("Years:", df["Year"].min(), "-", df["Year"].max())

df.head()

### Quick data quality check

Before analysis I want to see missing values, duplicates and how much coverage each feature has.

In [ ]:
quality = pd.DataFrame({
    "missing_n": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str),
}).sort_values("missing_pct", ascending=False)

print("Duplicated country-year-crop rows:", df.duplicated(["Code", "Year", "crop"]).sum())
quality

In [ ]:
df = add_features(df)

coverage = (
    df.groupby("crop")
      .agg(
          rows=("yield_t_ha", "size"),
          countries=("Code", "nunique"),
          first_year=("Year", "min"),
          last_year=("Year", "max"),
          median_yield=("yield_t_ha", "median"),
      )
      .sort_values("rows", ascending=False)
)
coverage

## 3. Business Questions

### 1. Which crops improved the most since 1990?

Before blaming climate for every change, I want to understand the long-term yield direction first.

In [ ]:
trend = (
    df.dropna(subset=["yield_t_ha"])
      .groupby(["crop", "Year"], as_index=False)["yield_t_ha"]
      .median()
)

plt.figure(figsize=(12, 6))
sns.lineplot(data=trend, x="Year", y="yield_t_ha", hue="crop", palette="Set2", linewidth=2)
plt.title("Median Crop Yield Across Countries (1990–2023)")
plt.xlabel("Year")
plt.ylabel("Yield (tonnes per hectare)")
plt.tight_layout()
plt.show()

### 2. How does temperature relate to crop yield?

A global scatter can be misleading because hot and cold countries grow different crops with different technology.

So I use **temperature deviation from each country's own average**. This asks a better question:

> When a country has a warmer-than-usual year, what happens to crop yield?

In [ ]:
sample = df.dropna(subset=["temp_deviation_c", "yield_t_ha"]).copy()

g = sns.lmplot(
    data=sample,
    x="temp_deviation_c",
    y="yield_t_ha",
    col="crop",
    col_wrap=3,
    scatter_kws={"alpha": 0.18, "s": 16},
    line_kws={"color": "green", "linewidth": 2},
    height=3.4,
    aspect=1.2,
)
g.set_axis_labels("Temperature deviation from country mean (°C)", "Yield (t/ha)")
g.fig.suptitle("Warmer-than-usual Years vs Crop Yield", y=1.03, fontsize=15)
plt.show()

### 3. Which crops look most temperature sensitive?

Here I estimate a simple descriptive slope for every crop. It is **not a causal effect**, but it is useful for ranking which crops deserve deeper attention.

In [ ]:
sensitivity = crop_temperature_sensitivity(df)

plt.figure(figsize=(10, 6))
colors = ["#f7786b" if v < 0 else "#99ff99"
          for v in sensitivity["yield_change_t_ha_per_1c_deviation"]]
ax = sns.barplot(
    data=sensitivity,
    x="crop",
    y="yield_change_t_ha_per_1c_deviation",
    hue="crop",
    palette=colors,
    legend=False,
)
plt.axhline(0, color="black", linewidth=1)
plt.title("Descriptive Yield Sensitivity to a +1°C Country-Level Deviation")
plt.xlabel("Crop")
plt.ylabel("Yield change (t/ha per +1°C deviation)")
plt.xticks(rotation=20)
label_bars(ax, decimals=3)
plt.tight_layout()
plt.show()

sensitivity

### 4. Is there one perfect temperature?

No magic exact number this time.

Instead of selecting the single temperature value with the highest observed yield, I use temperature ranges. This reduces the chance of calling one noisy observation "the optimum".

In [ ]:
binned = add_temperature_bins(df.dropna(subset=["temperature_c", "yield_t_ha"]), bins=8)

temp_summary = (
    binned.groupby(["crop", "temp_bin"], observed=True)["yield_t_ha"]
          .median()
          .reset_index()
)

g = sns.catplot(
    data=temp_summary,
    x="temp_bin",
    y="yield_t_ha",
    col="crop",
    col_wrap=2,
    kind="bar",
    palette="Greens",
    sharex=False,
    sharey=False,
    height=3.8,
    aspect=1.4,
)
g.set_axis_labels("Temperature range", "Median yield (t/ha)")
g.set_xticklabels(rotation=55, ha="right")
g.fig.suptitle("Yield by Temperature Range — Not One Magic Degree", y=1.02, fontsize=15)
plt.show()

### 5. Does more rain always mean better yield?

Agriculture needs water, but too little and too much can both be bad. I check the relationship by crop instead of assuming "more precipitation = more yield".

In [ ]:
rain = df.dropna(subset=["precipitation_mm", "yield_t_ha"]).copy()

g = sns.lmplot(
    data=rain,
    x="precipitation_mm",
    y="yield_t_ha",
    col="crop",
    col_wrap=3,
    lowess=True,
    scatter_kws={"alpha": 0.15, "s": 15},
    line_kws={"color": "#66b3ff", "linewidth": 2},
    height=3.4,
    aspect=1.2,
)
g.set_axis_labels("Annual precipitation (mm)", "Yield (t/ha)")
g.fig.suptitle("Precipitation vs Yield by Crop", y=1.03, fontsize=15)
plt.show()

### 6. Do irrigation and fertilizer appear to reduce climate risk?

This part is important for the client because climate itself is hard to control, but management decisions are actionable.

I compare yield and year-to-year yield volatility across management intensity levels.

In [ ]:
management = df.dropna(subset=["irrigated_land_pct", "fertilizer_kg_ha", "yield_t_ha"]).copy()

management["irrigation_group"] = pd.qcut(
    management["irrigated_land_pct"], q=4,
    labels=["Low", "Mid-low", "Mid-high", "High"],
    duplicates="drop",
)
management["fertilizer_group"] = pd.qcut(
    management["fertilizer_kg_ha"], q=4,
    labels=["Low", "Mid-low", "Mid-high", "High"],
    duplicates="drop",
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=management,
    x="irrigation_group",
    y="yield_t_ha",
    hue="irrigation_group",
    palette="Set2",
    legend=False,
    ax=axes[0],
    showfliers=False,
)
axes[0].set_title("Yield by Irrigation Intensity")
axes[0].set_xlabel("Irrigation group")
axes[0].set_ylabel("Yield (t/ha)")

sns.boxplot(
    data=management,
    x="fertilizer_group",
    y="yield_t_ha",
    hue="fertilizer_group",
    palette="Set2",
    legend=False,
    ax=axes[1],
    showfliers=False,
)
axes[1].set_title("Yield by Fertilizer Intensity")
axes[1].set_xlabel("Fertilizer group")
axes[1].set_ylabel("Yield (t/ha)")

plt.tight_layout()
plt.show()

### 7. Where is the climate risk highest?

For a simple business risk view I combine:

- **yield volatility** = unstable output
- **warming sensitivity** = yield falls in warmer-than-usual years

This is not an insurance-grade risk score. It is a prioritization tool for where the client should investigate first.

In [ ]:
risk_base = (
    df.dropna(subset=["yield_t_ha", "temp_deviation_c"])
      .groupby(["Code", "Entity", "crop"])
      .agg(
          yield_mean=("yield_t_ha", "mean"),
          yield_volatility=("yield_t_ha", "std"),
          observations=("yield_t_ha", "size"),
      )
      .reset_index()
)

# country-crop slope
slopes = []
for keys, part in df.dropna(subset=["yield_t_ha", "temp_deviation_c"]).groupby(["Code", "crop"]):
    if len(part) < 10 or part["temp_deviation_c"].nunique() < 3:
        continue
    slope = np.polyfit(part["temp_deviation_c"], part["yield_t_ha"], 1)[0]
    slopes.append({"Code": keys[0], "crop": keys[1], "temp_slope": slope})

risk = risk_base.merge(pd.DataFrame(slopes), on=["Code", "crop"], how="left")
risk = risk[risk["observations"] >= 10].copy()
risk["volatility_cv"] = risk["yield_volatility"] / risk["yield_mean"]
risk["warming_penalty"] = (-risk["temp_slope"]).clip(lower=0)

for c in ["volatility_cv", "warming_penalty"]:
    std = risk[c].std()
    risk[c + "_z"] = 0 if std == 0 else (risk[c] - risk[c].mean()) / std

risk["risk_score"] = risk[["volatility_cv_z", "warming_penalty_z"]].mean(axis=1)
risk.sort_values("risk_score", ascending=False).head(15)

### 8. Can we predict future-period yield better than a simple baseline?

I use a **time split** instead of a random split.

Training data is before 2018. Test data is 2018 onward. This is closer to the real use case: learn from the past and predict later years.

The model must also beat a very simple baseline: median yield for each crop from the training period.

In [ ]:
result = fit_time_split(df, split_year=2018, random_state=42)

pd.Series(result.metrics, name="value").to_frame()

In [ ]:
pred = result.predictions.copy()

plt.figure(figsize=(7, 7))
sns.scatterplot(
    data=pred.sample(min(4000, len(pred)), random_state=42),
    x="yield_t_ha",
    y="predicted_yield_t_ha",
    hue="crop",
    palette="Set2",
    alpha=0.6,
)
limit = max(pred["yield_t_ha"].max(), pred["predicted_yield_t_ha"].max())
plt.plot([0, limit], [0, limit], "--", color="black", linewidth=1)
plt.title("Actual vs Predicted Yield — 2018+ Holdout")
plt.xlabel("Actual yield (t/ha)")
plt.ylabel("Predicted yield (t/ha)")
plt.tight_layout()
plt.show()

### 9. Where does the model fail?

A good portfolio project should show model errors, not hide them.

I check which crops and countries have the highest absolute error so we know where the simple V1 model is not enough.

In [ ]:
error_by_crop = (
    pred.groupby("crop")["abs_error"]
        .agg(["mean", "median", "count"])
        .sort_values("mean", ascending=False)
)
error_by_crop

In [ ]:
error_by_country = (
    pred.groupby(["Code", "Entity"])["abs_error"]
        .agg(["mean", "median", "count"])
        .query("count >= 10")
        .sort_values("mean", ascending=False)
        .head(20)
)
error_by_country

## 4. Final Business Takeaways

After running the full notebook, summarize the final measured findings here.

The final answer should separate:

1. **What the data shows**
2. **What is only correlation / association**
3. **What the client can actually act on**
4. **Where more local agronomic data is needed**

The expected decision output is not "temperature is bad" or "fertilizer is good".

It should be more like:

> Climate exposure is crop-specific and country-specific. The client should prioritize crop-location combinations with both high yield volatility and a negative response to warmer-than-usual years, then evaluate irrigation and input strategy locally before changing operations.

## 5. Limitations

- Country averages hide local farm conditions.
- Annual climate averages hide heat waves, rainfall timing and growing-season effects.
- Fertilizer and irrigation are country-level indicators, not crop-specific treatments.
- Management variables are observational, so causal claims are not justified.
- The model is a decision-support baseline, not a farm-level forecasting system.

These limitations are important because they define the next version of the project instead of pretending V1 answers everything.

## Conclusion

This rebuild keeps the original business idea but changes the quality of the evidence.

The project now uses real public data, crop-specific yield series, climate variables, management indicators, a time-aware predictive baseline, explicit error analysis and clear limitations.

**Final client message:** climate risk cannot be removed, but it can be measured earlier and prioritized better.